# Testing the worm: even error

In this notebook, we implement several interactive tests that aim at checking that the worm algorithm we implemented samples correctly from the conditional distribution

In [15]:
from jax.sharding import Mesh, PartitionSpec, NamedSharding
from jax.typing import ArrayLike
import jax.numpy as jnp
from functools import partial
import numpy as np
from typing import Dict
from coherentinfo.moebius_two_odd_prime import MoebiusCodeTwoOddPrime
from coherentinfo.errormodel import ErrorModelLindbladTwoOddPrime
from coherentinfo.worm import (
    run_worm
)
import time
import json
import os
import jax
N_CPUS = os.cpu_count()
N_USED_CPUS = N_CPUS
print("Number of CPUs available: {}".format(N_CPUS))
print("Number of used CPUs: {}".format(N_USED_CPUS))
jax.config.update('jax_num_cpu_devices', N_USED_CPUS)
# jax.config.update("jax_log_compiles", True)
# Devices assumed by JAX
print(jax.devices())

Number of CPUs available: 16
Number of used CPUs: 16
[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7), CpuDevice(id=8), CpuDevice(id=9), CpuDevice(id=10), CpuDevice(id=11), CpuDevice(id=12), CpuDevice(id=13), CpuDevice(id=14), CpuDevice(id=15)]


# Test even error

The test consists in testing that the logical bit does not change if the error is initialized as even

In [16]:
length = 3
width = length
p = 3
moebius_setup = {"length": length, "width": width, "p": p}
num_samples = 1
num_worms = 100 * N_USED_CPUS  # 5000
burn_in_const = 5000
max_worm_const = 3000
scaling = "quadratic"
if scaling == "quadratic":
    burn_in_steps = length ** 2 * burn_in_const  # 50000
    max_worm_steps = burn_in_steps + length ** 2 * max_worm_const
elif scaling == "linear":
    burn_in_steps = length * burn_in_const  # 50000
    max_worm_steps = burn_in_steps + length * max_worm_const

gamma_t = 0.3

In [17]:
moebius_code = MoebiusCodeTwoOddPrime(
    length=length, width=width, d=2 * p)
em_lindblad = ErrorModelLindbladTwoOddPrime(
    moebius_code.num_edges, d=2 * p, gamma_t=gamma_t
)
print(f"Single error probabilities: {em_lindblad.get_probabilities()}")

Single error probabilities: [0.59932834 0.1721758  0.02563445 0.00505141 0.02563424 0.1721757 ]


In [45]:
# master_error_key = jax.random.PRNGKey(2)

initial_error_mod_d = jnp.zeros(moebius_code.num_edges, dtype=jnp.int32)
initial_error_mod_d = initial_error_mod_d.at[4].set(4)
initial_error_mod_d = initial_error_mod_d.at[1].set(2)
initial_error_mod_d = initial_error_mod_d.at[10].set(4)
initial_error_mod_d = initial_error_mod_d.at[3].set(4)
initial_error_mod_d = initial_error_mod_d.at[11].set(2)
initial_error_mod_2 = jnp.mod(initial_error_mod_d, 2)
initial_error_mod_p = jnp.mod(initial_error_mod_d, p)
initial_worm_error = jnp.vstack((initial_error_mod_2, initial_error_mod_p))

In [46]:
initial_error_mod_d

Array([0, 2, 0, 4, 4, 0, 0, 0, 0, 0, 4, 2, 0, 0, 0], dtype=int32)

In [47]:
syndrome_id = "plaquette"

if syndrome_id == "plaquette":
    h_mod_2 = moebius_code.h_x_mod_2
    h_mod_p = moebius_code.h_x_mod_p
    h_mod_d = moebius_code.h_x
    h_error_mod_p = moebius_code.h_z_mod_p
    compute_chi = moebius_code.compute_plaquette_syndrome_chi_x
elif syndrome_id == "vertex":
    h_mod_2 = moebius_code.h_z_mod_2
    h_mod_p = moebius_code.h_z_mod_p
    h_mod_d = moebius_code.h_z
    h_error_mod_p = moebius_code.h_x_mod_p
    compute_chi = moebius_code.compute_vertex_syndrome_chi_z


syndrome_mod_2 = jnp.mod(h_mod_2 @ initial_error_mod_2, 2)
syndrome_mod_p = jnp.mod(h_mod_p @ initial_error_mod_p, p)
syndrome = jnp.vstack([syndrome_mod_2, syndrome_mod_p])
syndrome_mod_d = jnp.mod(h_mod_d @ initial_error_mod_d, 2 * p)

initial_chi = jnp.mod(compute_chi(initial_error_mod_d)[-1], p)
num_stabs = h_mod_p.shape[0]

print(f"Initial chi: {initial_chi}")

Initial chi: 0


In [48]:
run_worm_partial = partial(
    run_worm,
    h_error_mod_p=h_error_mod_p,
    h_mod_p=h_mod_p,
    error_model=em_lindblad,
    compute_full_chi=compute_chi,
    num_stabs=num_stabs,
    burn_in_steps=burn_in_steps,
    max_worm_steps=max_worm_steps
)

# First over keys
run_worm_vmap = jax.vmap(run_worm_partial, in_axes=(None, 0))
# Then over initial errors
# run_worm_vmap = jax.vmap(run_worm_vmap, in_axes=(0, 0))
run_worm_jit = jax.jit(run_worm_vmap)

In [49]:
def get_chi_probs_and_binary_entropy(chi_vec, success_vec):
    # Number of successful worms
    num_success = jnp.sum(success_vec)
    # Sets simply to zero failed attempts so that they are not counted
    chi_vec_marked = jnp.where(success_vec, chi_vec, 0)
    p1 = jnp.sum(chi_vec_marked) / num_success
    p0 = 1 - p1
    binary_entropy = -jax.scipy.special.xlogy(p0, p0) / jnp.log(2)
    binary_entropy += -jax.scipy.special.xlogy(p1, p1) / jnp.log(2)
    return p0, p1, binary_entropy

In [50]:
master_worm_key = jax.random.PRNGKey(10)
worm_keys = jax.random.split(master_worm_key, num_worms)

devices = jax.devices()  # Assuming this returns 16 devices
mesh = Mesh(devices, ('batch',))

# sharding_for_keys = NamedSharding(mesh,  PartitionSpec('batch', None))
# worm_keys_sharded = jax.device_put(worm_keys, sharding_for_keys)

# 2. Define sharding: Split the 0th axis across 'batch', leave others whole
sharding_for_keys = NamedSharding(mesh,  PartitionSpec('batch', None))
worm_keys_sharded = jax.device_put(
    worm_keys, sharding_for_keys)

start = time.time()

new_worm_state = run_worm_jit(initial_worm_error, worm_keys_sharded)

p0, p1, entropy = get_chi_probs_and_binary_entropy(
    new_worm_state["chi"], new_worm_state["worm_success"])

end = time.time()

computation_time = end - start

print(f"p0: {p0}")
print(f"Computation time: {computation_time} s")

p0: 0.7962499856948853
Computation time: 14.246179819107056 s
